In [177]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
import requests
import time
import pandas as pd
import re
from lxml import html

# Método 1, Selenium (Muy lento)

In [178]:
columnas = ['Coche','Modelo','Precio', 'Año','Kilometraje (Km)','Tipo_Combustible','Caballos (CV)','Transmision','Lugar_Venta','Num_Puertas','Num_Plazas']
df = pd.DataFrame(columns=columnas)

In [179]:
links_coches = []

In [180]:
session = requests.Session()

In [181]:
for i in range(1,21):
    request = session.get(f'https://www.autoscout24.es/lst?atype=C&cy=E&damaged_listing=exclude&desc=0&page={i}&powertype=kw&search_id=2e3f6xpn1e9&sort=standard&source=listpage_pagination&ustate=N%2CU')
    soup = BeautifulSoup(request.text, 'html.parser')
    main = soup.find('main', class_='ListPage_main___0g2X')
    coches = main.find_all('article')
    for coche in coches:
        ruta = coche.find('a')
        links_coches.append('https://www.autoscout24.es'+ ruta['href'])
links_coches

['https://www.autoscout24.es/anuncios/volkswagen-polo-1-0-tsi-life-70kw-gasolina-blanco-a11276b5-1a07-4811-9c13-0b4b53d82d7f',
 'https://www.autoscout24.es/anuncios/volkswagen-tiguan-2-0tdi-dsg-life-110kw-diesel-negro-48a5cef4-f12a-47fe-a668-ab23e815e0e1',
 'https://www.autoscout24.es/anuncios/audi-q5-2-0tdi-s-line-quattro-ultra-s-tronic-120kw-diesel-blanco-c62ee7cb-4f74-4c2d-abba-40868e7d97c4',
 'https://www.autoscout24.es/anuncios/ford-kuga-monovolumen-automatico-de-5-puertas-diesel-negro-02e94916-9dcc-4df9-847c-abe949b8884c',
 'https://www.autoscout24.es/anuncios/ford-c-max-1-5tdci-titanium-120-diesel-gris-d9c766a6-8e8f-497f-8cb9-99e13c8b6091',
 'https://www.autoscout24.es/anuncios/audi-a6-2-7tdi-diesel-gris-ab37a9a6-0c66-4201-9039-a3e820e2a72f',
 'https://www.autoscout24.es/anuncios/ferrari-purosangue-6-5-gasolina-aea7c81b-3ec6-4529-a328-05095a3ee10c',
 'https://www.autoscout24.es/anuncios/renault-kadjar-1-3-tce-gpf-black-edition-edc-117kw-gasolina-rojo-cf3ba393-7458-4f8b-9d1a-c46f

In [182]:
options = Options()
service = Service(r"C:\Users\Nouh\Desktop\ProyectoATD\chromedriver.exe") 
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.autoscout24.es')
aceptar_cookies = WebDriverWait(driver, 1.5).until(
    EC.element_to_be_clickable((By.XPATH, '//*[@id="as24-cmp-popup"]/div/div[3]/button[2]'))
)
aceptar_cookies.click()
def scrap_coches(url):
    driver.get(url)
    coche = driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[2]/div[1]/div[2]/h1/div[1]/span[1]').text + ' ' + driver.find_element(By.XPATH, '//*[@id="__next"]/div/div/main/div[3]/div[2]/div[1]/div[2]/h1/div[1]/span[2]').text
    modelo = driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[2]/div[1]/div[2]/h1/div[2]').text
    precio = driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[2]/div[3]/div[1]/div/span').text[:-1]
    precios = re.findall('\d+', precio)
    precio_final = ''
    for precio_ in precios:
        precio_final += precio_
    precio_final += ' €'
    año = driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[3]/div/div[3]/div[4]').text
    km = (driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[3]/div/div[1]/div[4]').text.split(' '))[0]
    combustible = driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[3]/div/div[4]/div[4]').text
    caballos = (driver.find_element(By.XPATH,'//*[@id="technical-details-section"]/div/div[2]/dl/dd[1]').text.split('('))[-1][:-4]
    transmision = driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[3]/div/div[2]/div[4]').text
    lugar_venta = driver.find_element(By.XPATH,'//*[@id="__next"]/div/div/main/div[3]/div[2]/a').text
    num_puertas = driver.find_element(By.XPATH,'//*[@id="basic-details-section"]/div/div[2]/dl/dd[5]').text
    num_plazas = driver.find_element(By.XPATH,'//*[@id="basic-details-section"]/div/div[2]/dl/dd[4]').text
    df.loc[len(df)] = [coche,modelo,precio_final,año,km,combustible,caballos,transmision,lugar_venta,num_puertas,num_plazas]

In [184]:
for url in links_coches:
    coches_no_scrappeados = []
    try:
        scrap_coches(url)
    except:
        coches_no_scrappeados.append(url)
        print('Error para: ' + url)

Error para: https://www.autoscout24.es/anuncios/porsche-992-gt3-pdk-gasolina-gris-c1446076-cc17-457c-9e97-738c014531c3
Error para: https://www.autoscout24.es/anuncios/mini-cooper-se-electro-gasolina-azul-10595afd-698a-4236-a5d2-914ef170765b


In [186]:
df

,Coche,Modelo,Precio,Año,Kilometraje (Km),Tipo_Combustible,Caballos (CV),Transmision,Lugar_Venta,Num_Puertas,Num_Plazas
0,Volkswagen Polo,1.0 TSI Life 70kW,7900 €,10/2023,28.000,Gasolina,95,Manual,"SAN MARTIN DE LA VEGA, ES",5,5
1,Volkswagen Tiguan,2.0TDI DSG Life 110kW,8700 €,08/2023,39.000,Diésel,150,Automático,"SAN MARTIN DE LA VEGA, ES",5,5
2,Audi Q5,2.0TDI S line quattro-ultra S tronic 120kW,14900 €,11/2019,199.000,Diésel,163,Automático,"SAN MARTIN DE LA VEGA, ES",5,5
3,Ford Kuga,Monovolumen Automático de 5 Puertas,7450 €,05/2011,21.100,Diésel,140,Automático,"TORREVIEJA, ES",España,5
4,Ford C-Max,1.5TDCi Titanium 120,7990 €,02/2016,139.889,Diésel,120,Manual,"MADRID, ES",5,5
...,...,...,...,...,...,...,...,...,...,...,...
419,Kia Carens,1.6 GDi Concept,10300 €,06/2015,119.507,Gasolina,135,Manual,"LEGANES, ES",5,5
420,Mercedes-Benz A 170,CDI Avantgarde Aut.,2500 €,04/1999,276.207,Diésel,90,Automático,"ARGANDA DEL REY, ES",14926063,España
421,Porsche Macan,4,89900 €,01/2024,100,Electro/Gasolina,408,Automático,"SAN MIGUEL DE ABONA, ES",5,5
422,Opel Grandland X,1.5CDTi S&S Design Line 130,15900 €,10/2019,55.123,Diésel,131,Manual,"PONTEVEDRA, ES",5,5


In [185]:
df.to_excel('ScrapAutoScout24.xlsx', index=False)

In [187]:
df2 = df

In [190]:
df2

,Coche,Modelo,Precio,Año,Kilometraje (Km),Tipo_Combustible,Caballos (CV),Transmision,Lugar_Venta
0,Volkswagen Polo,1.0 TSI Life 70kW,7900 €,10/2023,28.000,Gasolina,95,Manual,"SAN MARTIN DE LA VEGA, ES"
1,Volkswagen Tiguan,2.0TDI DSG Life 110kW,8700 €,08/2023,39.000,Diésel,150,Automático,"SAN MARTIN DE LA VEGA, ES"
2,Audi Q5,2.0TDI S line quattro-ultra S tronic 120kW,14900 €,11/2019,199.000,Diésel,163,Automático,"SAN MARTIN DE LA VEGA, ES"
3,Ford Kuga,Monovolumen Automático de 5 Puertas,7450 €,05/2011,21.100,Diésel,140,Automático,"TORREVIEJA, ES"
4,Ford C-Max,1.5TDCi Titanium 120,7990 €,02/2016,139.889,Diésel,120,Manual,"MADRID, ES"
...,...,...,...,...,...,...,...,...,...
419,Kia Carens,1.6 GDi Concept,10300 €,06/2015,119.507,Gasolina,135,Manual,"LEGANES, ES"
420,Mercedes-Benz A 170,CDI Avantgarde Aut.,2500 €,04/1999,276.207,Diésel,90,Automático,"ARGANDA DEL REY, ES"
421,Porsche Macan,4,89900 €,01/2024,100,Electro/Gasolina,408,Automático,"SAN MIGUEL DE ABONA, ES"
422,Opel Grandland X,1.5CDTi S&S Design Line 130,15900 €,10/2019,55.123,Diésel,131,Manual,"PONTEVEDRA, ES"


In [191]:
df2.to_excel('ScrapAutoScout24_Arreglado.xlsx', index=False)

# Método 2, requests y uso de la API

In [ ]:
import re

In [ ]:
import pandas as pd
columnas = ['Coche','Modelo','Precio', 'Año','Kilometraje (Km)','Tipo_Combustible','Caballos (CV)','Transmision','Lugar_Venta','Enlace_Compra']
df = pd.DataFrame(columns=columnas)

## Verificamos que salen todos los coches:

In [ ]:
import requests
url = "https://www.autoscout24.es/_next/data/as24-search-funnel_main-706/lst.json"
payload = ""
headers = {
        "cookie": 'culture=es-ES; as24Visitor=3e9357bb-f078-4eb2-a3ee-8baf35987c3e; euconsent-v2=CQLlOMAQLlOMAGNAFBESBYFsAP_gAEPgAAAAKEBRpGZcTWFAIHp34KsAMAQX5RhBRMAhBACAk6AAyBMAIAwGEGAgIAAAJAAAABAAABABIAAACAAAAAAAQIAAIAAMAAAAgAIIICAAAEEAQAAAAAgIAAgAAAAAAEBEAAAAiAgAAAKAQNgAAAAAAQAAAAAAAAAAAAIAAAAwQAIAAAAAAAAAAQIAAAAAAAAAAAAAAAAAAAAAIAAQAAAAAAAAAAAABBQiByAAuACgAKgAcAA8ACAAEgAMgAaABEACOAEyAKQApgBVAC6AGIAPwAhABEACOAE0AJwAUYArQBhgDLAGiAOQAc4A7gB-wEHAQgAi0BHAEdAJKAYoAz4B1ADtgHtAP-Ai8BHoCZAFDgKPAU-AtQBbAC8wF_gMkAZYAy4B4oEEQIQgUIAGCQIwAFgAVAA4AB4AEAAL4AZABoADwAIgATAAqgBvAD8AISAQwBEgCOAE0AMMAZYA5wB3AD2gH4AfoBJQEiAKHAUeApEBbAC5AGSAMzAauBCEChA6BUAAsACoAHAAQAAvgBkAGgAPAAiABMACmAFUALoAYgA3gB-gEMARIAmgBRgDDAGiAOcAdwA9oB-AH6ARYAjoBJQDqAIvASIAmQBQ4CjwFsALkAZIAyoBlgDMwGrgOLAoQQgFAALACqAGIAN4AjgBzgDuAJSAdQBchKAmAAsADgAPAAiABMACqAGKAQwBEgCOAFGAPwA6gCLwEiAKPAWwAyQBlgEISkB0ABYAFQAOAAggBkAGgAPAAiABMACkAFUAMQAfoBDAESAKMAaIA54B-AH6ARYAjoBJQDqAIvASIAocBbAC5AGSAMsAhCWgBgDuAUOAzMAA.YAAAAAAAA4CA; cconsent-v2=%7B%22purpose%22%3A%7B%22legitimateInterests%22%3A%5B25%5D%2C%22consents%22%3A%5B26%5D%7D%2C%22vendor%22%3A%7B%22legitimateInterests%22%3A%5B10218%2C10441%2C11006%2C11005%2C11009%5D%2C%22consents%22%3A%5B10083%2C10211%2C10212%2C10218%2C10255%2C10304%2C10441%2C10442%2C10560%2C11002%2C11004%2C11005%2C11006%2C11009%5D%7D%7D; addtl_consent=1~70.89.326.486.1097.1329.2567.2822.3119; as24-cmp-signature=QVIpW%2BR860%2FMzmoo5n%2FO%2FF7LF5q9A1AbK9eh7rjL0io1Njs371OXB7jx8P%2BdauaBAzp3Stw%2FJtcQNfifsPyML9Wa7%2F33hPjfdwYwQEIJg16kgkTCnaibyQWDiVgCF6icDSLy5%2FpCNlT9ppJYuhCbgISgrtDHvh5u2X9ubrE2fXQ%3D; _gcl_gs=2.1.k1$i1737471295$u100268558; _gcl_au=1.1.821058145.1737471298; _utmz=other; _ga=GA1.1.912177948.1737471298; FPID=FPID2.2.vM8l6tZEagxXjxLrxTjWolPhBMzUKbbfJ8cS4iyVnFk%3D.1737471298; FPLC=uFgw1bMkKJRlQKZNSnsR%2F6dwSQ95p3tM9kV%2Fmh7ib1oGy7KQQ4bos%2FoIYCZ04WTZxxypYhaIr5Xqd5K4QkHvl%2FFzVK71s6grV3ffyGl6JzKqiv9Lek0zwlayiQwKZw%3D%3D; FPGCLAW=2.1.kCj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB$i1737471298; FPGCLGS=2.1.k1$i1737471295; FPAU=1.1.821058145.1737471298; _gtmeec=e30%3D; _fbp=fb.1.1737471298023.1196784184; _cq_duid=1.1737471298.GjVsfHJAwI6BCFtz; _cq_suid=1.1737471298.yBJrRTCgD9YwHQSS; _tt_enable_cookie=1; _ttp=AmpirlwnGLmGBC-sZCFHYlrfwzC.tt.1; ab_test_lp=%7B%7D; search-session=true; combined-leads=; as24-gtmSearchCrit=0100001-0500000:cy; ab_test_dp=%7B%7D; last-seen-classified-guid=eb32e652-74f2-4448-89f8-17d8f563c758; last-5-seen-classifieds=eb32e652-74f2-4448-89f8-17d8f563c758|fbd16d7c-e6d4-45b1-99bf-7498cbc06cc3|a11276b5-1a07-4811-9c13-0b4b53d82d7f|a11276b5-1a07-4811-9c13-0b4b53d82d7f; last-seen-listings=[{"id":"eb32e652-74f2-4448-89f8-17d8f563c758","price":27490,"mileage":45450,"power":300,"firstRegistrationYear":2019,"makeId":9,"modelId":75259,"fuelId":"2","latitude":41.54863,"longitude":2.10744,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737472501,"createdTimestampWithOffset":"2025-01-15T07:11:32.426Z"},{"id":"fbd16d7c-e6d4-45b1-99bf-7498cbc06cc3","price":4500,"mileage":169583,"power":50,"firstRegistrationYear":2012,"makeId":21,"modelId":18264,"fuelId":"D","latitude":41.94647,"longitude":2.78862,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737471562,"createdTimestampWithOffset":"2025-01-17T10:59:27.202Z"}]; _ga_D6E4J35ERN=GS1.1.1737471384.1.1.1737473819.0.0.0; last-search-feed=atype%3DC%26cy%3DE%26damaged_listing%3Dexclude%26desc%3D0%26powertype%3Dkw%26sort%3Dstandard%26ustate%3DN%252CU; as24-gtmlistpageno=1; _asse=cm:eyJzbSI6WyIyfDE3Mzc0NzEyOTgyOTN8MHwwfDI1MjE5NTN8Z29vZ2xlIiwxODAwNTQ1ODIwMjQ2XX0=; __rtbh.uid=%7B%22eventType%22%3A%22uid%22%2C%22id%22%3A%223e9357bb-f078-4eb2-a3ee-8baf35987c3e%22%2C%22expiryDate%22%3A%222026-01-21T15%3A37%3A00.362Z%22%7D; __rtbh.lid=%7B%22eventType%22%3A%22lid%22%2C%22id%22%3A%22RGPO2a4M1jldCy8TqwVw%22%2C%22expiryDate%22%3A%222026-01-21T15%3A37%3A00.362Z%22%7D; _ga_TX2QRVWP93=GS1.1.1737471298.1.1.1737473820.0.0.1889976552; _gcl_aw=GCL.1737473821.Cj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB; cto_bundle=8L8xGF9yU01RM2x2VHpOVnpuUm1kZXlIMDBDSjJYdDVhJTJCVVl0SGlTamRzUjRrRU5DdzNvQWpoZll5S2RuJTJCJTJCSGhkSjMlMkY2VGxWJTJCQmElMkY0aTY0RmRlaDRmS2YlMkZVcThvc2R3bHJ0Z0E2RFg2SExHYzNFNGszMkwlMkJwb0hnYmZuVWRmMGh2WFJTMGwwRXAzaVdvYSUyQmtNQjlpU2FUY3U3N3NPdXM5THVIRDlyQWJ0RGk0VlklM0Q; _cq_pxg=3|m0223363689649776992544532; _uetsid=af506730d80711efb0ed79ad094e47e1; _uetvid=6bc19800c55311efbe1e379bf37e672c; _ga_BGSHTTTQ7W=GS1.1.1737471298.1.1.1737473829.6.0.0',
        "accept": "*/*",
        "accept-language": "es-ES,es;q=0.9",
        "priority": "u=1, i",
        "referer": "https://www.autoscout24.es/lst?atype=C&cy=E&desc=0&page=1&search_id=ceb7anavqz&sort=standard&source=listpage_pagination&ustate=N%2CU",
        "sec-ch-ua": '"Not A(Brand";v="8", "Chromium";v="132", "Google Chrome";v="132"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-origin",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36",
        "x-nextjs-data": "1"
    }

for i in range(1,21):


    querystring = {"atype":"C","cy":"E","desc":"0","page":f"{i}","search_id":"ceb7anavqz","sort":"standard","source":"listpage_pagination","ustate":"N,U"}

   
    response = requests.request("GET", url, data=payload, headers=headers, params=querystring)

    response = response.json()

    coches = response['pageProps']['listings']

    print(f'Página {i}: \n {"*"*50}')
    for j in range(len(coches)-1):
        print(f'Coche nº{j+1}, {coches[j]["images"][0]}')

Página 1: 
 **************************************************
Coche nº1, https://prod.pictures.autoscout24.net/listing-images/5c98f10c-3c5f-45f0-8322-70c900232975_5e6da991-9395-4e48-8a3a-9a6b42e8e919.jpg/250x188.webp
Coche nº2, https://prod.pictures.autoscout24.net/listing-images/33f35f72-2a83-4f3e-b1f5-c73b838461fe_48925f8c-cf93-4d9d-8f4a-371188bb385d.jpg/250x188.webp
Coche nº3, https://prod.pictures.autoscout24.net/listing-images/71892d4f-4d9d-4963-a41f-4b29a05567af_ff80cb08-7bd0-4311-a82e-815ae4f776b3.jpg/250x188.webp
Coche nº4, https://prod.pictures.autoscout24.net/listing-images/9ef22b23-c31f-4294-a59f-171d609bfc70_d68edebc-c9b6-47d5-b1f3-8b8776048851.jpg/250x188.webp
Coche nº5, https://prod.pictures.autoscout24.net/listing-images/d389d426-5283-48fb-8b25-7c077e11a85f_b1a20994-4aa3-4afe-8245-28dfbe89889e.jpg/250x188.webp
Coche nº6, https://prod.pictures.autoscout24.net/listing-images/487e8e6a-815f-4fb2-a787-948eae15a0d4_e7b09391-0903-4802-90ed-ab69b83f7e17.jpg/250x188.webp
Coche n

IndexError: list index out of range

## Los añadimos a la base de Datos con una búsqueda general

In [ ]:
import requests
url = "https://www.autoscout24.es/_next/data/as24-search-funnel_main-706/lst.json"
payload = ""
headers = {
        "cookie": 'culture=es-ES; as24Visitor=3e9357bb-f078-4eb2-a3ee-8baf35987c3e; euconsent-v2=CQLlOMAQLlOMAGNAFBESBYFsAP_gAEPgAAAAKEBRpGZcTWFAIHp34KsAMAQX5RhBRMAhBACAk6AAyBMAIAwGEGAgIAAAJAAAABAAABABIAAACAAAAAAAQIAAIAAMAAAAgAIIICAAAEEAQAAAAAgIAAgAAAAAAEBEAAAAiAgAAAKAQNgAAAAAAQAAAAAAAAAAAAIAAAAwQAIAAAAAAAAAAQIAAAAAAAAAAAAAAAAAAAAAIAAQAAAAAAAAAAAABBQiByAAuACgAKgAcAA8ACAAEgAMgAaABEACOAEyAKQApgBVAC6AGIAPwAhABEACOAE0AJwAUYArQBhgDLAGiAOQAc4A7gB-wEHAQgAi0BHAEdAJKAYoAz4B1ADtgHtAP-Ai8BHoCZAFDgKPAU-AtQBbAC8wF_gMkAZYAy4B4oEEQIQgUIAGCQIwAFgAVAA4AB4AEAAL4AZABoADwAIgATAAqgBvAD8AISAQwBEgCOAE0AMMAZYA5wB3AD2gH4AfoBJQEiAKHAUeApEBbAC5AGSAMzAauBCEChA6BUAAsACoAHAAQAAvgBkAGgAPAAiABMACmAFUALoAYgA3gB-gEMARIAmgBRgDDAGiAOcAdwA9oB-AH6ARYAjoBJQDqAIvASIAmQBQ4CjwFsALkAZIAyoBlgDMwGrgOLAoQQgFAALACqAGIAN4AjgBzgDuAJSAdQBchKAmAAsADgAPAAiABMACqAGKAQwBEgCOAFGAPwA6gCLwEiAKPAWwAyQBlgEISkB0ABYAFQAOAAggBkAGgAPAAiABMACkAFUAMQAfoBDAESAKMAaIA54B-AH6ARYAjoBJQDqAIvASIAocBbAC5AGSAMsAhCWgBgDuAUOAzMAA.YAAAAAAAA4CA; cconsent-v2=%7B%22purpose%22%3A%7B%22legitimateInterests%22%3A%5B25%5D%2C%22consents%22%3A%5B26%5D%7D%2C%22vendor%22%3A%7B%22legitimateInterests%22%3A%5B10218%2C10441%2C11006%2C11005%2C11009%5D%2C%22consents%22%3A%5B10083%2C10211%2C10212%2C10218%2C10255%2C10304%2C10441%2C10442%2C10560%2C11002%2C11004%2C11005%2C11006%2C11009%5D%7D%7D; addtl_consent=1~70.89.326.486.1097.1329.2567.2822.3119; as24-cmp-signature=QVIpW%2BR860%2FMzmoo5n%2FO%2FF7LF5q9A1AbK9eh7rjL0io1Njs371OXB7jx8P%2BdauaBAzp3Stw%2FJtcQNfifsPyML9Wa7%2F33hPjfdwYwQEIJg16kgkTCnaibyQWDiVgCF6icDSLy5%2FpCNlT9ppJYuhCbgISgrtDHvh5u2X9ubrE2fXQ%3D; _gcl_gs=2.1.k1$i1737471295$u100268558; _gcl_au=1.1.821058145.1737471298; _utmz=other; _ga=GA1.1.912177948.1737471298; FPID=FPID2.2.vM8l6tZEagxXjxLrxTjWolPhBMzUKbbfJ8cS4iyVnFk%3D.1737471298; FPLC=uFgw1bMkKJRlQKZNSnsR%2F6dwSQ95p3tM9kV%2Fmh7ib1oGy7KQQ4bos%2FoIYCZ04WTZxxypYhaIr5Xqd5K4QkHvl%2FFzVK71s6grV3ffyGl6JzKqiv9Lek0zwlayiQwKZw%3D%3D; FPGCLAW=2.1.kCj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB$i1737471298; FPGCLGS=2.1.k1$i1737471295; FPAU=1.1.821058145.1737471298; _gtmeec=e30%3D; _fbp=fb.1.1737471298023.1196784184; _cq_duid=1.1737471298.GjVsfHJAwI6BCFtz; _cq_suid=1.1737471298.yBJrRTCgD9YwHQSS; _tt_enable_cookie=1; _ttp=AmpirlwnGLmGBC-sZCFHYlrfwzC.tt.1; ab_test_lp=%7B%7D; search-session=true; combined-leads=; as24-gtmSearchCrit=0100001-0500000:cy; ab_test_dp=%7B%7D; last-seen-classified-guid=eb32e652-74f2-4448-89f8-17d8f563c758; last-5-seen-classifieds=eb32e652-74f2-4448-89f8-17d8f563c758|fbd16d7c-e6d4-45b1-99bf-7498cbc06cc3|a11276b5-1a07-4811-9c13-0b4b53d82d7f|a11276b5-1a07-4811-9c13-0b4b53d82d7f; last-seen-listings=[{"id":"eb32e652-74f2-4448-89f8-17d8f563c758","price":27490,"mileage":45450,"power":300,"firstRegistrationYear":2019,"makeId":9,"modelId":75259,"fuelId":"2","latitude":41.54863,"longitude":2.10744,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737472501,"createdTimestampWithOffset":"2025-01-15T07:11:32.426Z"},{"id":"fbd16d7c-e6d4-45b1-99bf-7498cbc06cc3","price":4500,"mileage":169583,"power":50,"firstRegistrationYear":2012,"makeId":21,"modelId":18264,"fuelId":"D","latitude":41.94647,"longitude":2.78862,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737471562,"createdTimestampWithOffset":"2025-01-17T10:59:27.202Z"}]; _ga_D6E4J35ERN=GS1.1.1737471384.1.1.1737473819.0.0.0; last-search-feed=atype%3DC%26cy%3DE%26damaged_listing%3Dexclude%26desc%3D0%26powertype%3Dkw%26sort%3Dstandard%26ustate%3DN%252CU; as24-gtmlistpageno=1; _asse=cm:eyJzbSI6WyIyfDE3Mzc0NzEyOTgyOTN8MHwwfDI1MjE5NTN8Z29vZ2xlIiwxODAwNTQ1ODIwMjQ2XX0=; __rtbh.uid=%7B%22eventType%22%3A%22uid%22%2C%22id%22%3A%223e9357bb-f078-4eb2-a3ee-8baf35987c3e%22%2C%22expiryDate%22%3A%222026-01-21T15%3A37%3A00.362Z%22%7D; __rtbh.lid=%7B%22eventType%22%3A%22lid%22%2C%22id%22%3A%22RGPO2a4M1jldCy8TqwVw%22%2C%22expiryDate%22%3A%222026-01-21T15%3A37%3A00.362Z%22%7D; _ga_TX2QRVWP93=GS1.1.1737471298.1.1.1737473820.0.0.1889976552; _gcl_aw=GCL.1737473821.Cj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB; cto_bundle=8L8xGF9yU01RM2x2VHpOVnpuUm1kZXlIMDBDSjJYdDVhJTJCVVl0SGlTamRzUjRrRU5DdzNvQWpoZll5S2RuJTJCJTJCSGhkSjMlMkY2VGxWJTJCQmElMkY0aTY0RmRlaDRmS2YlMkZVcThvc2R3bHJ0Z0E2RFg2SExHYzNFNGszMkwlMkJwb0hnYmZuVWRmMGh2WFJTMGwwRXAzaVdvYSUyQmtNQjlpU2FUY3U3N3NPdXM5THVIRDlyQWJ0RGk0VlklM0Q; _cq_pxg=3|m0223363689649776992544532; _uetsid=af506730d80711efb0ed79ad094e47e1; _uetvid=6bc19800c55311efbe1e379bf37e672c; _ga_BGSHTTTQ7W=GS1.1.1737471298.1.1.1737473829.6.0.0',
        "accept": "*/*",
        "accept-language": "es-ES,es;q=0.9",
        "priority": "u=1, i",
        "referer": "https://www.autoscout24.es/lst?atype=C&cy=E&desc=0&page=1&search_id=ceb7anavqz&sort=standard&source=listpage_pagination&ustate=N%2CU",
        "sec-ch-ua": '"Not A(Brand";v="8", "Chromium";v="132", "Google Chrome";v="132"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-origin",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36",
        "x-nextjs-data": "1"
    }

for i in range(1,21):


    querystring = {"atype":"C","cy":"E","desc":"0","page":f"{i}","search_id":"ceb7anavqz","sort":"standard","source":"listpage_pagination","ustate":"N,U"}

   
    response = requests.request("GET", url, data=payload, headers=headers, params=querystring)

    response = response.json()

    coches = response['pageProps']['listings']

    print(f'Página {i}: \n {"*"*50}')
    for j in range(len(coches)):
        if response['pageProps']['listings'][j]['vehicle']['model']:
            coche = response['pageProps']['listings'][j]['vehicle']['make'] + ' ' + response['pageProps']['listings'][j]['vehicle']['model']
        else:
            coche = response['pageProps']['listings'][j]['vehicle']['make']
        modelo = response['pageProps']['listings'][j]['vehicle']['modelVersionInput']
        precio = response['pageProps']['listings'][j]['tracking']['price']
        año = response['pageProps']['listings'][j]['vehicleDetails'][2]['data']
        kilometraje = response['pageProps']['listings'][j]['vehicleDetails'][0]['data'][:-3]
        tipo_combustible = response['pageProps']['listings'][j]['vehicleDetails'][3]['data']
        caballos = response['pageProps']['listings'][j]['vehicleDetails'][4]['data'].split('(')[-1][:-4]
        transmision = response['pageProps']['listings'][j]['vehicleDetails'][1]['data']
        lugar_venta = response['pageProps']['listings'][j]['location']['city'] + ', ' + response['pageProps']['listings'][j]['location']['countryCode']
        enlace_venta = 'https://www.autoscout24.es/' + response['pageProps']['listings'][j]['url']
        df.loc[len(df)] = [coche,modelo,precio,año,kilometraje,tipo_combustible,caballos,transmision,lugar_venta,enlace_venta]
        print(f'Coche nº{j+1} insertado.')

Página 1: 
 **************************************************
Coche nº1 insertado.
Coche nº2 insertado.
Coche nº3 insertado.
Coche nº4 insertado.
Coche nº5 insertado.
Coche nº6 insertado.
Coche nº7 insertado.
Coche nº8 insertado.
Coche nº9 insertado.
Coche nº10 insertado.
Coche nº11 insertado.
Coche nº12 insertado.
Coche nº13 insertado.
Coche nº14 insertado.
Coche nº15 insertado.
Coche nº16 insertado.
Coche nº17 insertado.
Coche nº18 insertado.
Coche nº19 insertado.
Coche nº20 insertado.
Página 2: 
 **************************************************
Coche nº1 insertado.
Coche nº2 insertado.
Coche nº3 insertado.
Coche nº4 insertado.
Coche nº5 insertado.
Coche nº6 insertado.
Coche nº7 insertado.
Coche nº8 insertado.
Coche nº9 insertado.
Coche nº10 insertado.
Coche nº11 insertado.
Coche nº12 insertado.
Coche nº13 insertado.
Coche nº14 insertado.
Coche nº15 insertado.
Coche nº16 insertado.
Coche nº17 insertado.
Coche nº18 insertado.
Coche nº19 insertado.
Coche nº20 insertado.
Página 3: 
 

In [ ]:
df

,Coche,Modelo,Precio,Año,Kilometraje (Km),Tipo_Combustible,Caballos (CV),Transmision,Lugar_Venta,Enlace_Compra
0,Hyundai TUCSON,1.6 T 48V Maxx 4x4 DT,14500,05/2024,1,Gasolina,160,Automático,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-t...
1,SEAT Leon,1.9TDI Signo-a,3350,11/2004,203.150,Diésel,110,Manual,"ALICANTE, ES",https://www.autoscout24.es//anuncios/seat-leon...
2,Subaru Legacy,2.0D Premium,1800,12/2009,335.000,Diésel,150,Manual,"ASTIGARRAGA, ES",https://www.autoscout24.es//anuncios/subaru-le...
3,Opel Corsa,1.2T XHL S/S GS 100,13990,03/2024,12.986,Gasolina,101,Manual,"MADRID, ES",https://www.autoscout24.es//anuncios/opel-cors...
4,Nissan Qashqai,Familiar Manual de 5 Puertas,6499,06/2016,214.000,Diésel,110,Manual,"SAN PEDRO DEL PINATAR, ES",https://www.autoscout24.es//anuncios/nissan-qa...
...,...,...,...,...,...,...,...,...,...,...
395,Opel Corsa,Nuevo GS 1.2T XHL MT6 S/S 100 CV (74kW),13941,01/2024,14.487,Gasolina,101,Manual,"OLIAS DEL REY, ES",https://www.autoscout24.es//anuncios/opel-cors...
396,Audi A4,2.0TDI Advanced edition DPF 150,12300,11/2014,199.000,Diésel,150,Manual,"ORIHUELA COSTA, ES",https://www.autoscout24.es//anuncios/audi-a4-2...
397,Dacia Sandero,1.2I 75CV 5P 5PL,4300,06/2011,116.773,Gasolina,75,Manual,"VILABLAREIX, ES",https://www.autoscout24.es//anuncios/dacia-san...
398,BMW 118,118d,19990,08/2020,59.828,Diésel,150,Manual,"MADRID, ES",https://www.autoscout24.es//anuncios/bmw-118-1...


## Añadimos coches dedicados a busquedas de cada marca para equilibrar la base de datos (Tener mas coches)

### Verificar que los encontramos

In [ ]:
marcas = ['mercedes-benz', 'bmw', 'audi', 'porsche', 'volkswagen', 'toyota', 'ford', 'ferrari', 'volvo', 'hyundai']
for marca in marcas:
    print(f'Inserción de coches de {marca.upper()}')
    url = f"https://www.autoscout24.es/_next/data/as24-search-funnel_main-706/lst/{marca}.json"


    payload = ""
    headers = {
        "cookie": 'culture=es-ES; as24Visitor=3e9357bb-f078-4eb2-a3ee-8baf35987c3e; euconsent-v2=CQLlOMAQLlOMAGNAFBESBYFsAP_gAEPgAAAAKEBRpGZcTWFAIHp34KsAMAQX5RhBRMAhBACAk6AAyBMAIAwGEGAgIAAAJAAAABAAABABIAAACAAAAAAAQIAAIAAMAAAAgAIIICAAAEEAQAAAAAgIAAgAAAAAAEBEAAAAiAgAAAKAQNgAAAAAAQAAAAAAAAAAAAIAAAAwQAIAAAAAAAAAAQIAAAAAAAAAAAAAAAAAAAAAIAAQAAAAAAAAAAAABBQiByAAuACgAKgAcAA8ACAAEgAMgAaABEACOAEyAKQApgBVAC6AGIAPwAhABEACOAE0AJwAUYArQBhgDLAGiAOQAc4A7gB-wEHAQgAi0BHAEdAJKAYoAz4B1ADtgHtAP-Ai8BHoCZAFDgKPAU-AtQBbAC8wF_gMkAZYAy4B4oEEQIQgUIAGCQIwAFgAVAA4AB4AEAAL4AZABoADwAIgATAAqgBvAD8AISAQwBEgCOAE0AMMAZYA5wB3AD2gH4AfoBJQEiAKHAUeApEBbAC5AGSAMzAauBCEChA6BUAAsACoAHAAQAAvgBkAGgAPAAiABMACmAFUALoAYgA3gB-gEMARIAmgBRgDDAGiAOcAdwA9oB-AH6ARYAjoBJQDqAIvASIAmQBQ4CjwFsALkAZIAyoBlgDMwGrgOLAoQQgFAALACqAGIAN4AjgBzgDuAJSAdQBchKAmAAsADgAPAAiABMACqAGKAQwBEgCOAFGAPwA6gCLwEiAKPAWwAyQBlgEISkB0ABYAFQAOAAggBkAGgAPAAiABMACkAFUAMQAfoBDAESAKMAaIA54B-AH6ARYAjoBJQDqAIvASIAocBbAC5AGSAMsAhCWgBgDuAUOAzMAA.YAAAAAAAA4CA; cconsent-v2=%7B%22purpose%22%3A%7B%22legitimateInterests%22%3A%5B25%5D%2C%22consents%22%3A%5B26%5D%7D%2C%22vendor%22%3A%7B%22legitimateInterests%22%3A%5B10218%2C10441%2C11006%2C11005%2C11009%5D%2C%22consents%22%3A%5B10083%2C10211%2C10212%2C10218%2C10255%2C10304%2C10441%2C10442%2C10560%2C11002%2C11004%2C11005%2C11006%2C11009%5D%7D%7D; addtl_consent=1~70.89.326.486.1097.1329.2567.2822.3119; as24-cmp-signature=QVIpW%2BR860%2FMzmoo5n%2FO%2FF7LF5q9A1AbK9eh7rjL0io1Njs371OXB7jx8P%2BdauaBAzp3Stw%2FJtcQNfifsPyML9Wa7%2F33hPjfdwYwQEIJg16kgkTCnaibyQWDiVgCF6icDSLy5%2FpCNlT9ppJYuhCbgISgrtDHvh5u2X9ubrE2fXQ%3D; _gcl_gs=2.1.k1$i1737471295$u100268558; _gcl_au=1.1.821058145.1737471298; _utmz=other; _ga=GA1.1.912177948.1737471298; FPID=FPID2.2.vM8l6tZEagxXjxLrxTjWolPhBMzUKbbfJ8cS4iyVnFk%3D.1737471298; FPLC=uFgw1bMkKJRlQKZNSnsR%2F6dwSQ95p3tM9kV%2Fmh7ib1oGy7KQQ4bos%2FoIYCZ04WTZxxypYhaIr5Xqd5K4QkHvl%2FFzVK71s6grV3ffyGl6JzKqiv9Lek0zwlayiQwKZw%3D%3D; FPGCLAW=2.1.kCj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB$i1737471298; FPGCLGS=2.1.k1$i1737471295; FPAU=1.1.821058145.1737471298; _gtmeec=e30%3D; _fbp=fb.1.1737471298023.1196784184; _cq_duid=1.1737471298.GjVsfHJAwI6BCFtz; _cq_suid=1.1737471298.yBJrRTCgD9YwHQSS; _tt_enable_cookie=1; _ttp=AmpirlwnGLmGBC-sZCFHYlrfwzC.tt.1; ab_test_lp=%7B%7D; search-session=true; combined-leads=; ab_test_dp=%7B%7D; as24-gtmlistpageno=20; last-seen-classified-guid=08178fd1-f1d8-4d47-87c9-f333051a0b5b; last-5-seen-classifieds=08178fd1-f1d8-4d47-87c9-f333051a0b5b|5c98f10c-3c5f-45f0-8322-70c900232975|08178fd1-f1d8-4d47-87c9-f333051a0b5b|5c98f10c-3c5f-45f0-8322-70c900232975|08178fd1-f1d8-4d47-87c9-f333051a0b5b; last-seen-listings=[{"id":"08178fd1-f1d8-4d47-87c9-f333051a0b5b","price":1995,"mileage":290000,"power":55,"firstRegistrationYear":2008,"makeId":54,"modelId":1918,"fuelId":"D","latitude":36.7202,"longitude":-4.5034,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737476778,"createdTimestampWithOffset":"2025-01-13T11:29:48.078Z"},{"id":"5c98f10c-3c5f-45f0-8322-70c900232975","price":14500,"mileage":1,"power":118,"firstRegistrationYear":2024,"makeId":33,"modelId":18493,"fuelId":"B","latitude":40.2078,"longitude":-3.56854,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737476777,"createdTimestampWithOffset":"2025-01-20T16:49:38.04Z"},{"id":"6077ca7a-aff8-4ae6-a7d4-9d30f4827aa9","price":11850,"mileage":29999,"power":74,"firstRegistrationYear":2022,"makeId":54,"modelId":1918,"fuelId":"B","latitude":43.26323,"longitude":-3.01431,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737475103,"createdTimestampWithOffset":"2025-01-13T15:01:05.989Z"},{"id":"240fac2d-b419-436f-a29d-573481ae54ef","price":57990,"mileage":85000,"power":287,"firstRegistrationYear":2020,"makeId":47,"modelId":20922,"fuelId":"2","latitude":43.21789,"longitude":-2.81428,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737474241,"createdTimestampWithOffset":"2024-12-11T17:32:36.554Z"},{"id":"eb32e652-74f2-4448-89f8-17d8f563c758","price":27490,"mileage":45450,"power":300,"firstRegistrationYear":2019,"makeId":9,"modelId":75259,"fuelId":"2","latitude":41.54863,"longitude":2.10744,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737472501,"createdTimestampWithOffset":"2025-01-15T07:11:32.426Z"},{"id":"fbd16d7c-e6d4-45b1-99bf-7498cbc06cc3","price":4500,"mileage":169583,"power":50,"firstRegistrationYear":2012,"makeId":21,"modelId":18264,"fuelId":"D","latitude":41.94647,"longitude":2.78862,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737471562,"createdTimestampWithOffset":"2025-01-17T10:59:27.202Z"}]; _ga_D6E4J35ERN=GS1.1.1737471384.1.1.1737476843.0.0.0; last-search-feed=atype%3DC%26cy%3DE%26damaged_listing%3Dexclude%26desc%3D0%26powertype%3Dkw%26sort%3Dstandard%26ustate%3DN%252CU%26mmm%3D47%7C%7C; as24-gtmSearchCrit=0020001-0050000:ma|cy; _asse=cm:eyJzbSI6WyIyfDE3Mzc0NzEyOTgyOTN8MHwwfDU1NDYxMjB8Z29vZ2xlIiwxODAwNTQ4ODQ0NDEzXX0=; __rtbh.uid=%7B%22eventType%22%3A%22uid%22%2C%22id%22%3A%223e9357bb-f078-4eb2-a3ee-8baf35987c3e%22%2C%22expiryDate%22%3A%222026-01-21T16%3A27%3A24.501Z%22%7D; __rtbh.lid=%7B%22eventType%22%3A%22lid%22%2C%22id%22%3A%22RGPO2a4M1jldCy8TqwVw%22%2C%22expiryDate%22%3A%222026-01-21T16%3A27%3A24.501Z%22%7D; _gcl_aw=GCL.1737476845.Cj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB; _ga_TX2QRVWP93=GS1.1.1737471298.1.1.1737476844.0.0.1889976552; _uetsid=af506730d80711efb0ed79ad094e47e1; _uetvid=6bc19800c55311efbe1e379bf37e672c; cto_bundle=6g78wF9yU01RM2x2VHpOVnpuUm1kZXlIMDBDRFpRaUYwYmpJMU42eXhMUkRpRGNJd2ZXdmx0cmE1MkxQbmJZb2o2ZVFnU1FVMnppZ0JPcFJPNndKNnN3UVZqOGlGREoyS1VCTWxEdUhiQnhuU0Z4QWxyZUU4UjglMkZlaFhQMU1GOUUlMkZrcHBWSG5KMU11bEIwN1pnZkdaY2pNc05GSEFRaGpLOGxsanVmaUV5dUUyM0NNJTNE; _cq_pxg=3|9748405544657; _ga_BGSHTTTQ7W=GS1.1.1737471298.1.1.1737476850.53.0.0',
        "accept": "*/*",
        "accept-language": "es-ES,es;q=0.9",
        "priority": "u=1, i",
        "referer": "https://www.autoscout24.es/lst/mercedes-benz?atype=C&cy=E&damaged_listing=exclude&desc=0&powertype=kw&search_id=klmoxucd60&sort=standard&source=listpage_pagination&ustate=N%2CU",
        "sec-ch-ua": '"Not A(Brand";v="8", "Chromium";v="132", "Google Chrome";v="132"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-origin",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36",
        "x-nextjs-data": "1"
    }
    try:
        for i in range(1,21):
            querystring = {"atype":"C","cy":"E","damaged_listing":"exclude","desc":"0","powertype":"kw","search_id":"klmoxucd60","sort":"standard","source":"listpage_pagination","ustate":"N,U","page":f"{i}","slug":"{marca}"}

            response = requests.request("GET", url, data=payload, headers=headers, params=querystring)

            coches = response.json()['pageProps']['listings']
            print(f'Página {i}: \n {"*"*50}')
            for j in range(len(coches)):
                print(f'Coche nº{j+1} de {marca} encontrado: {coches[j]["images"][0]}')
    except:
        pass

Inserción de coches de MERCEDES-BENZ
Página 1: 
 **************************************************
Coche nº1 de mercedes-benz encontrado: https://prod.pictures.autoscout24.net/listing-images/44878bd7-9182-4953-a04d-9ef8ff38eb44_367daefb-16f3-4e48-9119-a2e4cd7fd5ac.jpg/250x188.webp
Coche nº2 de mercedes-benz encontrado: https://prod.pictures.autoscout24.net/listing-images/60d05724-f266-45cf-9616-e989413651ac_fcad2b88-752e-43b8-a0db-a92ac56a030c.jpg/250x188.webp
Coche nº3 de mercedes-benz encontrado: https://prod.pictures.autoscout24.net/listing-images/0f2dc4b2-4527-40dc-8aa5-0cf1dae372ee_d9cfce1d-5572-4b38-b4ca-e788e9c9e49a.jpg/250x188.webp
Coche nº4 de mercedes-benz encontrado: https://prod.pictures.autoscout24.net/listing-images/7f27e76f-adc4-4b94-a81c-239dcd869dc1_5923ff09-2b21-4e4d-99cd-ba2edf78e96e.jpg/250x188.webp
Coche nº5 de mercedes-benz encontrado: https://prod.pictures.autoscout24.net/listing-images/dc759861-8912-4360-8c64-df7e4a961806_2bd16efd-a32a-460b-8f99-fc027ff795b1.jp

### Los añadimos a la base de datos

In [ ]:
marcas = ['mercedes-benz', 'bmw', 'audi', 'porsche', 'volkswagen', 'toyota', 'ford', 'ferrari', 'volvo', 'hyundai']
for marca in marcas:
    print(f'Inserción de coches de {marca.upper()}')
    url = f"https://www.autoscout24.es/_next/data/as24-search-funnel_main-706/lst/{marca}.json"


    payload = ""
    headers = {
        "cookie": 'culture=es-ES; as24Visitor=3e9357bb-f078-4eb2-a3ee-8baf35987c3e; euconsent-v2=CQLlOMAQLlOMAGNAFBESBYFsAP_gAEPgAAAAKEBRpGZcTWFAIHp34KsAMAQX5RhBRMAhBACAk6AAyBMAIAwGEGAgIAAAJAAAABAAABABIAAACAAAAAAAQIAAIAAMAAAAgAIIICAAAEEAQAAAAAgIAAgAAAAAAEBEAAAAiAgAAAKAQNgAAAAAAQAAAAAAAAAAAAIAAAAwQAIAAAAAAAAAAQIAAAAAAAAAAAAAAAAAAAAAIAAQAAAAAAAAAAAABBQiByAAuACgAKgAcAA8ACAAEgAMgAaABEACOAEyAKQApgBVAC6AGIAPwAhABEACOAE0AJwAUYArQBhgDLAGiAOQAc4A7gB-wEHAQgAi0BHAEdAJKAYoAz4B1ADtgHtAP-Ai8BHoCZAFDgKPAU-AtQBbAC8wF_gMkAZYAy4B4oEEQIQgUIAGCQIwAFgAVAA4AB4AEAAL4AZABoADwAIgATAAqgBvAD8AISAQwBEgCOAE0AMMAZYA5wB3AD2gH4AfoBJQEiAKHAUeApEBbAC5AGSAMzAauBCEChA6BUAAsACoAHAAQAAvgBkAGgAPAAiABMACmAFUALoAYgA3gB-gEMARIAmgBRgDDAGiAOcAdwA9oB-AH6ARYAjoBJQDqAIvASIAmQBQ4CjwFsALkAZIAyoBlgDMwGrgOLAoQQgFAALACqAGIAN4AjgBzgDuAJSAdQBchKAmAAsADgAPAAiABMACqAGKAQwBEgCOAFGAPwA6gCLwEiAKPAWwAyQBlgEISkB0ABYAFQAOAAggBkAGgAPAAiABMACkAFUAMQAfoBDAESAKMAaIA54B-AH6ARYAjoBJQDqAIvASIAocBbAC5AGSAMsAhCWgBgDuAUOAzMAA.YAAAAAAAA4CA; cconsent-v2=%7B%22purpose%22%3A%7B%22legitimateInterests%22%3A%5B25%5D%2C%22consents%22%3A%5B26%5D%7D%2C%22vendor%22%3A%7B%22legitimateInterests%22%3A%5B10218%2C10441%2C11006%2C11005%2C11009%5D%2C%22consents%22%3A%5B10083%2C10211%2C10212%2C10218%2C10255%2C10304%2C10441%2C10442%2C10560%2C11002%2C11004%2C11005%2C11006%2C11009%5D%7D%7D; addtl_consent=1~70.89.326.486.1097.1329.2567.2822.3119; as24-cmp-signature=QVIpW%2BR860%2FMzmoo5n%2FO%2FF7LF5q9A1AbK9eh7rjL0io1Njs371OXB7jx8P%2BdauaBAzp3Stw%2FJtcQNfifsPyML9Wa7%2F33hPjfdwYwQEIJg16kgkTCnaibyQWDiVgCF6icDSLy5%2FpCNlT9ppJYuhCbgISgrtDHvh5u2X9ubrE2fXQ%3D; _gcl_gs=2.1.k1$i1737471295$u100268558; _gcl_au=1.1.821058145.1737471298; _utmz=other; _ga=GA1.1.912177948.1737471298; FPID=FPID2.2.vM8l6tZEagxXjxLrxTjWolPhBMzUKbbfJ8cS4iyVnFk%3D.1737471298; FPLC=uFgw1bMkKJRlQKZNSnsR%2F6dwSQ95p3tM9kV%2Fmh7ib1oGy7KQQ4bos%2FoIYCZ04WTZxxypYhaIr5Xqd5K4QkHvl%2FFzVK71s6grV3ffyGl6JzKqiv9Lek0zwlayiQwKZw%3D%3D; FPGCLAW=2.1.kCj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB$i1737471298; FPGCLGS=2.1.k1$i1737471295; FPAU=1.1.821058145.1737471298; _gtmeec=e30%3D; _fbp=fb.1.1737471298023.1196784184; _cq_duid=1.1737471298.GjVsfHJAwI6BCFtz; _cq_suid=1.1737471298.yBJrRTCgD9YwHQSS; _tt_enable_cookie=1; _ttp=AmpirlwnGLmGBC-sZCFHYlrfwzC.tt.1; ab_test_lp=%7B%7D; search-session=true; combined-leads=; ab_test_dp=%7B%7D; as24-gtmlistpageno=20; last-seen-classified-guid=08178fd1-f1d8-4d47-87c9-f333051a0b5b; last-5-seen-classifieds=08178fd1-f1d8-4d47-87c9-f333051a0b5b|5c98f10c-3c5f-45f0-8322-70c900232975|08178fd1-f1d8-4d47-87c9-f333051a0b5b|5c98f10c-3c5f-45f0-8322-70c900232975|08178fd1-f1d8-4d47-87c9-f333051a0b5b; last-seen-listings=[{"id":"08178fd1-f1d8-4d47-87c9-f333051a0b5b","price":1995,"mileage":290000,"power":55,"firstRegistrationYear":2008,"makeId":54,"modelId":1918,"fuelId":"D","latitude":36.7202,"longitude":-4.5034,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737476778,"createdTimestampWithOffset":"2025-01-13T11:29:48.078Z"},{"id":"5c98f10c-3c5f-45f0-8322-70c900232975","price":14500,"mileage":1,"power":118,"firstRegistrationYear":2024,"makeId":33,"modelId":18493,"fuelId":"B","latitude":40.2078,"longitude":-3.56854,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737476777,"createdTimestampWithOffset":"2025-01-20T16:49:38.04Z"},{"id":"6077ca7a-aff8-4ae6-a7d4-9d30f4827aa9","price":11850,"mileage":29999,"power":74,"firstRegistrationYear":2022,"makeId":54,"modelId":1918,"fuelId":"B","latitude":43.26323,"longitude":-3.01431,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737475103,"createdTimestampWithOffset":"2025-01-13T15:01:05.989Z"},{"id":"240fac2d-b419-436f-a29d-573481ae54ef","price":57990,"mileage":85000,"power":287,"firstRegistrationYear":2020,"makeId":47,"modelId":20922,"fuelId":"2","latitude":43.21789,"longitude":-2.81428,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737474241,"createdTimestampWithOffset":"2024-12-11T17:32:36.554Z"},{"id":"eb32e652-74f2-4448-89f8-17d8f563c758","price":27490,"mileage":45450,"power":300,"firstRegistrationYear":2019,"makeId":9,"modelId":75259,"fuelId":"2","latitude":41.54863,"longitude":2.10744,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737472501,"createdTimestampWithOffset":"2025-01-15T07:11:32.426Z"},{"id":"fbd16d7c-e6d4-45b1-99bf-7498cbc06cc3","price":4500,"mileage":169583,"power":50,"firstRegistrationYear":2012,"makeId":21,"modelId":18264,"fuelId":"D","latitude":41.94647,"longitude":2.78862,"isSmyle":false,"isLeasingMarktPremium":null,"timestamp":1737471562,"createdTimestampWithOffset":"2025-01-17T10:59:27.202Z"}]; _ga_D6E4J35ERN=GS1.1.1737471384.1.1.1737476843.0.0.0; last-search-feed=atype%3DC%26cy%3DE%26damaged_listing%3Dexclude%26desc%3D0%26powertype%3Dkw%26sort%3Dstandard%26ustate%3DN%252CU%26mmm%3D47%7C%7C; as24-gtmSearchCrit=0020001-0050000:ma|cy; _asse=cm:eyJzbSI6WyIyfDE3Mzc0NzEyOTgyOTN8MHwwfDU1NDYxMjB8Z29vZ2xlIiwxODAwNTQ4ODQ0NDEzXX0=; __rtbh.uid=%7B%22eventType%22%3A%22uid%22%2C%22id%22%3A%223e9357bb-f078-4eb2-a3ee-8baf35987c3e%22%2C%22expiryDate%22%3A%222026-01-21T16%3A27%3A24.501Z%22%7D; __rtbh.lid=%7B%22eventType%22%3A%22lid%22%2C%22id%22%3A%22RGPO2a4M1jldCy8TqwVw%22%2C%22expiryDate%22%3A%222026-01-21T16%3A27%3A24.501Z%22%7D; _gcl_aw=GCL.1737476845.Cj0KCQiAqL28BhCrARIsACYJvkfPMHhV2Ftqz-hKzSmZYV95NYZkDexW04gJGN5QzYeS6FLzbI6b7rUaAvvOEALw_wcB; _ga_TX2QRVWP93=GS1.1.1737471298.1.1.1737476844.0.0.1889976552; _uetsid=af506730d80711efb0ed79ad094e47e1; _uetvid=6bc19800c55311efbe1e379bf37e672c; cto_bundle=6g78wF9yU01RM2x2VHpOVnpuUm1kZXlIMDBDRFpRaUYwYmpJMU42eXhMUkRpRGNJd2ZXdmx0cmE1MkxQbmJZb2o2ZVFnU1FVMnppZ0JPcFJPNndKNnN3UVZqOGlGREoyS1VCTWxEdUhiQnhuU0Z4QWxyZUU4UjglMkZlaFhQMU1GOUUlMkZrcHBWSG5KMU11bEIwN1pnZkdaY2pNc05GSEFRaGpLOGxsanVmaUV5dUUyM0NNJTNE; _cq_pxg=3|9748405544657; _ga_BGSHTTTQ7W=GS1.1.1737471298.1.1.1737476850.53.0.0',
        "accept": "*/*",
        "accept-language": "es-ES,es;q=0.9",
        "priority": "u=1, i",
        "referer": "https://www.autoscout24.es/lst/mercedes-benz?atype=C&cy=E&damaged_listing=exclude&desc=0&powertype=kw&search_id=klmoxucd60&sort=standard&source=listpage_pagination&ustate=N%2CU",
        "sec-ch-ua": '"Not A(Brand";v="8", "Chromium";v="132", "Google Chrome";v="132"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-origin",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36",
        "x-nextjs-data": "1"
    }
    try:
        for i in range(1,21):
            querystring = {"atype":"C","cy":"E","damaged_listing":"exclude","desc":"0","powertype":"kw","search_id":"klmoxucd60","sort":"standard","source":"listpage_pagination","ustate":"N,U","page":f"{i}","slug":"{marca}"}

            response = requests.request("GET", url, data=payload, headers=headers, params=querystring)

            coches = response.json()['pageProps']['listings']
            print(f'Página {i}: \n {"*"*50}')
            for j in range(len(coches)):
                if coches[j]['vehicle']['model']:
                    coche = coches[j]['vehicle']['make'] + ' ' + coches[j]['vehicle']['model']
                else:
                    coche = coches[j]['vehicle']['make']
                modelo = coches[j]['vehicle']['modelVersionInput']
                precio = coches[j]['tracking']['price']
                año = coches[j]['vehicleDetails'][2]['data']
                kilometraje = coches[j]['vehicleDetails'][0]['data'][:-3]
                tipo_combustible = coches[j]['vehicleDetails'][3]['data']
                caballos = coches[j]['vehicleDetails'][4]['data'].split('(')[-1][:-4]
                transmision = coches[j]['vehicleDetails'][1]['data']
                lugar_venta = coches[j]['location']['city'] + ', ' + coches[j]['location']['countryCode']
                enlace_venta = 'https://www.autoscout24.es/' + coches[j]['url']
                df.loc[len(df)] = [coche,modelo,precio,año,kilometraje,tipo_combustible,caballos,transmision,lugar_venta,enlace_venta]
                print(f'Coche nº{j+1} de {marca} insertado: {coches[j]["images"][0]}')
    except:
        pass

Inserción de coches de MERCEDES-BENZ
Página 1: 
 **************************************************
Coche nº1 de mercedes-benz insertado: https://prod.pictures.autoscout24.net/listing-images/44878bd7-9182-4953-a04d-9ef8ff38eb44_367daefb-16f3-4e48-9119-a2e4cd7fd5ac.jpg/250x188.webp
Coche nº2 de mercedes-benz insertado: https://prod.pictures.autoscout24.net/listing-images/60d05724-f266-45cf-9616-e989413651ac_fcad2b88-752e-43b8-a0db-a92ac56a030c.jpg/250x188.webp
Coche nº3 de mercedes-benz insertado: https://prod.pictures.autoscout24.net/listing-images/0f2dc4b2-4527-40dc-8aa5-0cf1dae372ee_d9cfce1d-5572-4b38-b4ca-e788e9c9e49a.jpg/250x188.webp
Coche nº4 de mercedes-benz insertado: https://prod.pictures.autoscout24.net/listing-images/7f27e76f-adc4-4b94-a81c-239dcd869dc1_5923ff09-2b21-4e4d-99cd-ba2edf78e96e.jpg/250x188.webp
Coche nº5 de mercedes-benz insertado: https://prod.pictures.autoscout24.net/listing-images/dc759861-8912-4360-8c64-df7e4a961806_2bd16efd-a32a-460b-8f99-fc027ff795b1.jpg/250

In [ ]:
df

,Coche,Modelo,Precio,Año,Kilometraje (Km),Tipo_Combustible,Caballos (CV),Transmision,Lugar_Venta,Enlace_Compra
0,Hyundai TUCSON,1.6 T 48V Maxx 4x4 DT,14500,05/2024,1,Gasolina,160,Automático,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-t...
1,SEAT Leon,1.9TDI Signo-a,3350,11/2004,203.150,Diésel,110,Manual,"ALICANTE, ES",https://www.autoscout24.es//anuncios/seat-leon...
2,Subaru Legacy,2.0D Premium,1800,12/2009,335.000,Diésel,150,Manual,"ASTIGARRAGA, ES",https://www.autoscout24.es//anuncios/subaru-le...
3,Opel Corsa,1.2T XHL S/S GS 100,13990,03/2024,12.986,Gasolina,101,Manual,"MADRID, ES",https://www.autoscout24.es//anuncios/opel-cors...
4,Nissan Qashqai,Familiar Manual de 5 Puertas,6499,06/2016,214.000,Diésel,110,Manual,"SAN PEDRO DEL PINATAR, ES",https://www.autoscout24.es//anuncios/nissan-qa...
...,...,...,...,...,...,...,...,...,...,...
3750,Hyundai BAYON,1.2 MPI Maxx,7900,06/2021,78.000,Gasolina,84,Manual,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-b...
3751,Hyundai BAYON,1.2 MPI Klass,15900,06/2024,6.914,Gasolina,105,Manual,"PALENCIA, ES",https://www.autoscout24.es//anuncios/hyundai-b...
3752,Hyundai i20,i20 1.0 TGDI Tecno LE 2C 100 Tecno LE 2C,11500,04/2019,19.000,Gasolina,99,Manual,"MONCOFAR, ES",https://www.autoscout24.es//anuncios/hyundai-i...
3753,Hyundai ACCENT,1.5CRDI GLS 5p.,1000,07/2004,81.335,Diésel,82,Manual,"BILBAO, ES",https://www.autoscout24.es//anuncios/hyundai-a...


# El hecho de que los km tenga punto nos da problemas así que realizamos una transformacion

In [ ]:
def quitar_puntos(x):
    cadena_final = x
    if '.' in x:
        cadena = x.split('.')
        cadena_final = ''
        for valor in cadena:
            cadena_final += valor
    try:
        int(cadena_final)
        return int(cadena_final)
    except:
        return 1

In [ ]:
df['Kilometraje (Km)'] = df['Kilometraje (Km)'].apply(quitar_puntos)

In [ ]:
df

,Coche,Modelo,Precio,Año,Kilometraje (Km),Tipo_Combustible,Caballos (CV),Transmision,Lugar_Venta,Enlace_Compra
0,Hyundai TUCSON,1.6 T 48V Maxx 4x4 DT,14500,05/2024,1,Gasolina,160,Automático,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-t...
1,SEAT Leon,1.9TDI Signo-a,3350,11/2004,203150,Diésel,110,Manual,"ALICANTE, ES",https://www.autoscout24.es//anuncios/seat-leon...
2,Subaru Legacy,2.0D Premium,1800,12/2009,335000,Diésel,150,Manual,"ASTIGARRAGA, ES",https://www.autoscout24.es//anuncios/subaru-le...
3,Opel Corsa,1.2T XHL S/S GS 100,13990,03/2024,12986,Gasolina,101,Manual,"MADRID, ES",https://www.autoscout24.es//anuncios/opel-cors...
4,Nissan Qashqai,Familiar Manual de 5 Puertas,6499,06/2016,214000,Diésel,110,Manual,"SAN PEDRO DEL PINATAR, ES",https://www.autoscout24.es//anuncios/nissan-qa...
...,...,...,...,...,...,...,...,...,...,...
3750,Hyundai BAYON,1.2 MPI Maxx,7900,06/2021,78000,Gasolina,84,Manual,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-b...
3751,Hyundai BAYON,1.2 MPI Klass,15900,06/2024,6914,Gasolina,105,Manual,"PALENCIA, ES",https://www.autoscout24.es//anuncios/hyundai-b...
3752,Hyundai i20,i20 1.0 TGDI Tecno LE 2C 100 Tecno LE 2C,11500,04/2019,19000,Gasolina,99,Manual,"MONCOFAR, ES",https://www.autoscout24.es//anuncios/hyundai-i...
3753,Hyundai ACCENT,1.5CRDI GLS 5p.,1000,07/2004,81335,Diésel,82,Manual,"BILBAO, ES",https://www.autoscout24.es//anuncios/hyundai-a...


# Además, para facilitar el sistema de búsqueda puede que nos interese tener una columna para la marca y otra para el modelo, así que:

In [ ]:
df[['Marca', 'Modelo']] = df['Coche'].str.split(' ', n=1, expand=True)
df


,Coche,Modelo,Precio,Año,Kilometraje (Km),Tipo_Combustible,Caballos (CV),Transmision,Lugar_Venta,Enlace_Compra,Marca
0,Hyundai TUCSON,TUCSON,14500,05/2024,1,Gasolina,160,Automático,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-t...,Hyundai
1,SEAT Leon,Leon,3350,11/2004,203150,Diésel,110,Manual,"ALICANTE, ES",https://www.autoscout24.es//anuncios/seat-leon...,SEAT
2,Subaru Legacy,Legacy,1800,12/2009,335000,Diésel,150,Manual,"ASTIGARRAGA, ES",https://www.autoscout24.es//anuncios/subaru-le...,Subaru
3,Opel Corsa,Corsa,13990,03/2024,12986,Gasolina,101,Manual,"MADRID, ES",https://www.autoscout24.es//anuncios/opel-cors...,Opel
4,Nissan Qashqai,Qashqai,6499,06/2016,214000,Diésel,110,Manual,"SAN PEDRO DEL PINATAR, ES",https://www.autoscout24.es//anuncios/nissan-qa...,Nissan
...,...,...,...,...,...,...,...,...,...,...,...
3750,Hyundai BAYON,BAYON,7900,06/2021,78000,Gasolina,84,Manual,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-b...,Hyundai
3751,Hyundai BAYON,BAYON,15900,06/2024,6914,Gasolina,105,Manual,"PALENCIA, ES",https://www.autoscout24.es//anuncios/hyundai-b...,Hyundai
3752,Hyundai i20,i20,11500,04/2019,19000,Gasolina,99,Manual,"MONCOFAR, ES",https://www.autoscout24.es//anuncios/hyundai-i...,Hyundai
3753,Hyundai ACCENT,ACCENT,1000,07/2004,81335,Diésel,82,Manual,"BILBAO, ES",https://www.autoscout24.es//anuncios/hyundai-a...,Hyundai


In [ ]:
df = df.drop(columns=['Coche'])


In [ ]:
df = df[['Marca', 'Modelo', 'Precio','Año','Kilometraje (Km)', 'Tipo_Combustible', 'Caballos (CV)', 'Transmision', 'Lugar_Venta', 'Enlace_Compra']]


In [ ]:
df

,Marca,Modelo,Precio,Año,Kilometraje (Km),Tipo_Combustible,Caballos (CV),Transmision,Lugar_Venta,Enlace_Compra
0,Hyundai,TUCSON,14500,05/2024,1,Gasolina,160,Automático,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-t...
1,SEAT,Leon,3350,11/2004,203150,Diésel,110,Manual,"ALICANTE, ES",https://www.autoscout24.es//anuncios/seat-leon...
2,Subaru,Legacy,1800,12/2009,335000,Diésel,150,Manual,"ASTIGARRAGA, ES",https://www.autoscout24.es//anuncios/subaru-le...
3,Opel,Corsa,13990,03/2024,12986,Gasolina,101,Manual,"MADRID, ES",https://www.autoscout24.es//anuncios/opel-cors...
4,Nissan,Qashqai,6499,06/2016,214000,Diésel,110,Manual,"SAN PEDRO DEL PINATAR, ES",https://www.autoscout24.es//anuncios/nissan-qa...
...,...,...,...,...,...,...,...,...,...,...
3750,Hyundai,BAYON,7900,06/2021,78000,Gasolina,84,Manual,"SAN MARTIN DE LA VEGA, ES",https://www.autoscout24.es//anuncios/hyundai-b...
3751,Hyundai,BAYON,15900,06/2024,6914,Gasolina,105,Manual,"PALENCIA, ES",https://www.autoscout24.es//anuncios/hyundai-b...
3752,Hyundai,i20,11500,04/2019,19000,Gasolina,99,Manual,"MONCOFAR, ES",https://www.autoscout24.es//anuncios/hyundai-i...
3753,Hyundai,ACCENT,1000,07/2004,81335,Diésel,82,Manual,"BILBAO, ES",https://www.autoscout24.es//anuncios/hyundai-a...


In [ ]:
df.to_csv('AutoScout24.csv', index=False)